In [1]:
# STEP 2 | CUSTOMER LEVEL AGGREGATION

# Build one row per user_id with the five behaviour metrics

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Assigning path
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [4]:
# Load the master file

In [5]:
df_master = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'instacart_master.pkl')
)

print(f"Master loaded: {df_master.shape}")
print()

# ------------------------------------------------------------
# 2A — BUILD CUSTOMER-LEVEL TABLE
# ------------------------------------------------------------
# One groupby pass on user_id computes all five metrics.
# 
# - unique_orders: nunique(order_id) — true distinct order count
# - unique_products: nunique(product_id) — basket diversity
# - avg_product_price: mean(prices) at line level (weighted by purchase frequency)
# - total_reordered_items: sum(reordered) — count of reorder events
# - total_line_items: count of rows — used as denominator for reorder_rate
#
# reorder_rate is computed AFTER the groupby because it requires
# both the sum and the count to be available as columns.

df_customers_agg = df_master.groupby('user_id').agg(
    unique_orders         = ('order_id',   'nunique'),
    unique_products       = ('product_id', 'nunique'),
    avg_product_price     = ('prices',     'mean'),
    total_reordered_items = ('reordered',  'sum'),
    total_line_items      = ('reordered',  'count')   # count of rows per user
).reset_index()

# Compute reorder_rate as a derived column.
# Definition: fraction of all product lines that were reorders.
df_customers_agg['reorder_rate'] = (
    df_customers_agg['total_reordered_items'] / df_customers_agg['total_line_items']
)

print(f"✅ Customer aggregation complete: {df_customers_agg.shape}")
print()

# ------------------------------------------------------------
# 2B — VALIDATION
# ------------------------------------------------------------
# Hard requirements from the brief:
# - One row per user_id
# - User count matches the customers source file (206,209)
# - No nulls in any computed metric
# - reorder_rate must be between 0 and 1 inclusive

# Row count must equal unique users in the master file
assert df_customers_agg.shape[0] == df_master['user_id'].nunique(), \
    "Customer table row count does not match unique users in master"

# user_id must be unique in the customer table
assert df_customers_agg['user_id'].duplicated().sum() == 0

# Customer count must match the original customers.csv
# (206,209 — confirmed in profiling, all users have prior orders)
assert df_customers_agg.shape[0] == 206209, \
    f"Expected 206,209 customers, got {df_customers_agg.shape[0]}"

# No nulls in any aggregated column
assert df_customers_agg.isnull().sum().sum() == 0, \
    "Unexpected nulls in customer aggregation"

# reorder_rate must be a valid proportion
assert df_customers_agg['reorder_rate'].between(0, 1).all(), \
    "reorder_rate outside [0, 1] — check arithmetic"

print("✅ All Step 2 assertions passed")
print()

# ------------------------------------------------------------
# 2C — DESCRIPTIVE OUTPUT
# ------------------------------------------------------------
# Print summary stats so we can sanity-check the distributions
# before they feed into percentile ranks in Step 3.

print("Customer-level metric distributions:")
print(df_customers_agg[
    ['unique_orders', 'unique_products', 'avg_product_price',
     'reorder_rate', 'total_reordered_items']
].describe().round(3))
print()

print("Sample (first 10 customers):")
print(df_customers_agg.head(10).to_string())
print()

print(f"✅ STEP 2 COMPLETE — df_customers_agg ready for Step 3")
print(f"   Shape: {df_customers_agg.shape}")

Master loaded: (32398592, 21)

✅ Customer aggregation complete: (206209, 7)

✅ All Step 2 assertions passed

Customer-level metric distributions:
       unique_orders  unique_products  avg_product_price  reorder_rate  \
count     206209.000       206209.000         206209.000    206209.000   
mean          15.589           64.482              7.752         0.432   
std           16.654           56.556              1.039         0.212   
min            1.000            1.000              1.000         0.000   
25%            5.000           25.000              7.223         0.268   
50%            9.000           48.000              7.791         0.429   
75%           19.000           86.000              8.330         0.596   
max           99.000          725.000             23.200         0.990   

       total_reordered_items  
count             206209.000  
mean                  92.633  
std                  158.214  
min                    0.000  
25%                   10.000  
5

In [6]:
# How many products have absurd prices?
print("Products with price >= $100:")
print(df_master[df_master['prices'] >= 100][
    ['product_id', 'product_name', 'department_name', 'prices']
].drop_duplicates().sort_values('prices', ascending=False).head(20))
print()

print("Distribution of prices in master:")
print(df_master['prices'].describe(percentiles=[.5, .9, .99, .999, .9999]))
print()

print("Count of transactions at each high price tier:")
print(f"  prices >= $50:    {(df_master['prices'] >= 50).sum():,}")
print(f"  prices >= $100:   {(df_master['prices'] >= 100).sum():,}")
print(f"  prices >= $1000:  {(df_master['prices'] >= 1000).sum():,}")
print(f"  prices >= $10000: {(df_master['prices'] >= 10000).sum():,}")

Products with price >= $100:
        product_id                      product_name department_name   prices
128909       33664             2 % Reduced Fat  Milk      dairy eggs  99999.0
1576         21553  Lowfat 2% Milkfat Cottage Cheese      dairy eggs  14900.0

Distribution of prices in master:
count     3.240372e+07
mean      1.198041e+01
std       4.956642e+02
min       1.000000e+00
50%       7.400000e+00
90%       1.360000e+01
99%       1.820000e+01
99.9%     2.460000e+01
99.99%    1.490000e+04
max       9.999900e+04
Name: prices, dtype: float64

Count of transactions at each high price tier:
  prices >= $50:    5,127
  prices >= $100:   5,127
  prices >= $1000:  5,127
  prices >= $10000: 5,127
